# Dataset Preparation — All 4 Schemes
Generates Schemes A, B, C, D from QWS dataset.
Steps: Load → Clean → Split → Normalize → Generate Lists → Apply Scheme Weights
All schemes share same node pools and list compositions — only ranking order differs.

In [ ]:
# CELL 1: Imports and seeds
import pandas as pd
import numpy as np
import random
import os

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('Imports done!')

In [ ]:
# CELL 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/'
QWS_RAW   = BASE_PATH + 'qws2.csv'

for scheme in ['A', 'B', 'C', 'D']:
    os.makedirs(BASE_PATH + f'Scheme_{scheme}/', exist_ok=True)

print('Paths set!')

In [ ]:
# CELL 3: Define scheme weights
SCHEMES = {
    'A': {
        'description': 'Equal weights',
        'weights': {
            'Response_Time_norm': 0.2,
            'Availability_norm':  0.2,
            'Throughput_norm':    0.2,
            'Reliability_norm':   0.2,
            'Latency_norm':       0.2
        }
    },
    'B': {
        'description': 'Latency-dominant',
        'weights': {
            'Response_Time_norm': 0.3,
            'Availability_norm':  0.1,
            'Throughput_norm':    0.1,
            'Reliability_norm':   0.1,
            'Latency_norm':       0.4
        }
    },
    'C': {
        'description': 'Availability-dominant',
        'weights': {
            'Response_Time_norm': 0.0667,
            'Availability_norm':  0.5,
            'Throughput_norm':    0.0667,
            'Reliability_norm':   0.3,
            'Latency_norm':       0.0667
        }
    },
    'D': {
        'description': 'Throughput-dominant',
        'weights': {
            'Response_Time_norm': 0.3,
            'Availability_norm':  0.0667,
            'Throughput_norm':    0.5,
            'Reliability_norm':   0.0667,
            'Latency_norm':       0.0667
        }
    }
}

for s, info in SCHEMES.items():
    total = sum(info['weights'].values())
    print(f'Scheme {s} ({info["description"]}): weights sum = {total:.4f}')

In [ ]:
# CELL 4: Step 1 — Load and clean raw QWS data

columns = [
    'Response_Time', 'Availability', 'Throughput', 'Successability',
    'Reliability', 'Compliance', 'Best_Practices', 'Latency',
    'Documentation', 'Service_Name', 'WSDL_Address'
]

try:
    df = pd.read_csv(QWS_RAW, names=columns, skiprows=20, header=None)
except:
    df = pd.read_csv(QWS_RAW, names=columns, skiprows=20, header=None,
                     on_bad_lines='skip', engine='python')

# Select 5 core QoS attributes
QOS_COLS = ['Response_Time', 'Availability', 'Throughput', 'Reliability', 'Latency']
df = df[QOS_COLS + ['Service_Name']].copy()

# Convert to numeric
for col in QOS_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove missing values
df = df.dropna().reset_index(drop=True)
df['node_id'] = range(1, len(df) + 1)

print(f'Total nodes after cleaning: {len(df)}')
print(df[QOS_COLS].describe().round(2))

In [ ]:
# CELL 5: Step 2 — Split nodes 80/10/10
# CRITICAL: random_state=42 for reproducibility

df_shuffled = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

n = len(df_shuffled)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

train_pool = df_shuffled.iloc[:train_end].copy().reset_index(drop=True)
val_pool   = df_shuffled.iloc[train_end:val_end].copy().reset_index(drop=True)
test_pool  = df_shuffled.iloc[val_end:].copy().reset_index(drop=True)

print(f'Train pool: {len(train_pool)} nodes')
print(f'Val pool:   {len(val_pool)} nodes')
print(f'Test pool:  {len(test_pool)} nodes')

# Verify no overlap
train_ids = set(train_pool['node_id'])
val_ids   = set(val_pool['node_id'])
test_ids  = set(test_pool['node_id'])
print(f'\nTrain ∩ Val:  {len(train_ids & val_ids)} ')
print(f'Train ∩ Test: {len(train_ids & test_ids)} ')
print(f'Val ∩ Test:   {len(val_ids & test_ids)} ')

In [ ]:
# CELL 6: Step 3 — Normalize using train pool statistics ONLY
# Lower is better: Response_Time, Latency → inverted normalization
# Higher is better: Availability, Throughput, Reliability → standard normalization

HIGHER_IS_BETTER = ['Availability', 'Throughput', 'Reliability']
LOWER_IS_BETTER  = ['Response_Time', 'Latency']
NORM_COLS        = ['Response_Time_norm', 'Availability_norm', 'Throughput_norm',
                    'Reliability_norm', 'Latency_norm']

train_min = train_pool[QOS_COLS].min()
train_max = train_pool[QOS_COLS].max()

def normalize(pool_df):
    df = pool_df.copy()
    for col in HIGHER_IS_BETTER:
        df[f'{col}_norm'] = (df[col] - train_min[col]) / (train_max[col] - train_min[col])
    for col in LOWER_IS_BETTER:
        df[f'{col}_norm'] = (train_max[col] - df[col]) / (train_max[col] - train_min[col])
    df[NORM_COLS] = df[NORM_COLS].clip(0, 1)
    return df

train_pool = normalize(train_pool)
val_pool   = normalize(val_pool)
test_pool  = normalize(test_pool)

print('Normalization done using train pool statistics.')
print('Norm value ranges:')
for col in NORM_COLS:
    print(f'  {col}: [{train_pool[col].min():.4f}, {train_pool[col].max():.4f}]')

In [ ]:
# CELL 7: Step 4 — Generate ranking lists

def create_ranking_lists(pool_df, num_lists, scheme_weights,
                         nodes_per_list=10, seed_offset=0):
    all_lists = []

    for i in range(num_lists):
        # Sample nodes — replace=False ensures unique nodes within list
        sampled = pool_df.sample(
            n=nodes_per_list,
            replace=False,
            random_state=RANDOM_SEED + i + seed_offset
        ).copy()

        # Compute scheme-specific QoS score
        sampled['qos_score'] = sum(
            sampled[col] * w for col, w in scheme_weights.items()
        )

        # Sort by qos_score descending — ground truth ranking
        sampled = sampled.sort_values('qos_score', ascending=False).reset_index(drop=True)

        # Randomise display_id — prevents model from memorising position
        node_labels = [f'Node_{j+1}' for j in range(nodes_per_list)]
        random.seed(RANDOM_SEED + i + seed_offset)
        random.shuffle(node_labels)

        sampled['display_id'] = node_labels
        sampled['list_id']    = i
        sampled['list_rank']  = range(1, nodes_per_list + 1)

        all_lists.append(sampled)

    return pd.concat(all_lists, ignore_index=True)

print('List generation function defined.')

In [ ]:
# CELL 8: Step 5 — Generate all 4 schemes
# Seed offsets ensure no overlap in random states between splits

LIST_COUNTS  = {'train': len(train_pool), 'val': len(val_pool), 'test': len(test_pool)}
SEED_OFFSETS = {'train': 0, 'val': 5000, 'test': 10000}
POOLS        = {'train': train_pool, 'val': val_pool, 'test': test_pool}
FILE_NAMES   = {'train': 'obj3_train', 'val': 'obj3_val', 'test': 'obj3_test'}

print(f'List counts: train={LIST_COUNTS["train"]}, val={LIST_COUNTS["val"]}, test={LIST_COUNTS["test"]}')

for scheme, info in SCHEMES.items():
    print(f'\n{"="*60}')
    print(f'Generating Scheme {scheme}: {info["description"]}')
    print(f'{"="*60}')

    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    for split in ['train', 'val', 'test']:
        df_lists = create_ranking_lists(
            pool_df        = POOLS[split],
            num_lists      = LIST_COUNTS[split],
            scheme_weights = info['weights'],
            seed_offset    = SEED_OFFSETS[split]
        )

        filename = f'{FILE_NAMES[split]}_{scheme}.csv'
        filepath = BASE_PATH + f'Scheme_{scheme}/' + filename
        df_lists.to_csv(filepath, index=False)
        print(f'  Saved: {filename} — {df_lists["list_id"].nunique()} lists, {len(df_lists)} rows')

In [ ]:
# CELL 9: Verification
print('='*60)
print('VERIFICATION')
print('='*60)

for scheme in ['A', 'B', 'C', 'D']:
    test_file  = BASE_PATH + f'Scheme_{scheme}/obj3_test_{scheme}.csv'
    train_file = BASE_PATH + f'Scheme_{scheme}/obj3_train_{scheme}.csv'
    test_df    = pd.read_csv(test_file)
    train_df   = pd.read_csv(train_file)

    # Check no duplicate nodes within any list
    has_duplicates = False
    for lid in test_df['list_id'].unique()[:10]:
        group = test_df[test_df['list_id'] == lid]
        if group['node_id'].nunique() < len(group):
            has_duplicates = True
            break

    # Check no train/test overlap
    train_nodes = set(train_df['node_id'].unique())
    test_nodes  = set(test_df['node_id'].unique())
    no_overlap  = len(train_nodes & test_nodes) == 0

    # Check norm values in [0,1]
    norm_ok = all(
        test_df[col].between(0, 1).all()
        for col in NORM_COLS
    )

    print(f'\nScheme {scheme} ({SCHEMES[scheme]["description"]}):')
    print(f'  No duplicate nodes within list: {not has_duplicates} - PASS' if not has_duplicates else f'  No duplicate nodes within list: {not has_duplicates} - FAIL')
    print(f'  No train/test overlap:          {no_overlap} - PASS' if no_overlap else f'  No train/test overlap:          {no_overlap} - FAIL')
    print(f'  Norm values in [0,1]:           {norm_ok} - PASS' if norm_ok else f'  Norm values in [0,1]:           {norm_ok} - FAIL')
    print(f'  Train lists: {train_df["list_id"].nunique()} | Test lists: {test_df["list_id"].nunique()}')

# Check rankings differ across schemes
print('\n=== Rankings differ across schemes (list_id=0)? ===')
rankings = {}
for scheme in ['A', 'B', 'C', 'D']:
    test_file = BASE_PATH + f'Scheme_{scheme}/obj3_test_{scheme}.csv'
    df = pd.read_csv(test_file)
    rankings[scheme] = df[df['list_id']==0].sort_values('list_rank')['node_id'].tolist()

for s1, s2 in [('A','B'), ('A','C'), ('A','D')]:
    diff = rankings[s1] != rankings[s2]
    print(f'  Scheme {s1} vs {s2}: differ = {diff} - PASS' if diff else f'  Scheme {s1} vs {s2}: differ = {diff} - FAIL')

In [ ]:
# CELL 10: Summary

print('='*60)
print('DATASET GENERATION COMPLETE')
print('='*60)
print()
for scheme, info in SCHEMES.items():
    print(f'Scheme {scheme} ({info["description"]}):')
    for split in ['train', 'val', 'test']:
        f = BASE_PATH + f'Scheme_{scheme}/obj3_{split}_{scheme}.csv'
        df = pd.read_csv(f)
        print(f'  obj3_{split}_{scheme}.csv — {df["list_id"].nunique()} lists, {len(df)} rows')
    print()
print('All schemes share same node pools.')
print('Rankings differ per scheme based on weights.')
print('replace=False: unique nodes within each list.')